<a href="https://colab.research.google.com/github/yashb98/AI-ML-interview-Prep/blob/main/Interview_Prep_Day_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Implementing Scaled Dot-Product Atyention using Numpy ( no Pytorch )

In [11]:
import numpy as np

def scaled_dot_product_attention(query, key, value, mask=None):
  """
  Compute Scaled dot-product Attention

  Dimensions:
      batch_size: B
      num_heads: H - ignored here for simplicity, or trated part if B
      seq_len: L
      head_dim: D
  """

  # Extract the dimensionality of the heads (d, k)
  # query_states shape : [batch_size, seq_len, head_dim]
  head_dim = query.shape[-1]

  #Calculate raw attention scores ( logits)
  # [B, L, D] @ [B, D, L] -> [B, L, L]
  attention_scores = np.matmul(query, key.swapaxes(-2, -1))

  # Apply the scaling factor
  attention_mask = attention_scores / np.sqrt(head_dim)

  # Apply attention_mask if provided (e.g. for padding or casual masking)
  if attention_mask is not None:
    attention_scores += attention_mask

  # Softmax for probability distribution
  # Subtracting np.max is a numerical stability trick to prevent overflow
  shifted_logits = attention_scores - np.max(attention_scores, axis=-1, keepdims=True)
  attention_probs = np.exp(shifted_logits) /np.sum(np.exp(shifted_logits), axis=-1, keepdims=True)

  #Context layer: apply weights to the value states
  # [B, L, L] @ [B, L, D] -> [B, L, D]
  context_layer =np.matmul(attention_probs, value)

  return context_layer, attention_probs


# Standard embedding input [batch, seq_ken, model_dim]
hidden_states = np.random.randn(1, 16, 512)

# Learned weight matrices ( in a real model, these are trained)
W_q = np.random.randn(512, 512)
W_k = np.random.randn(512, 512)
W_v = np.random.randn(512, 512)

# Project to get our states
query_states = np.matmul(hidden_states, W_q)
key_states = np.matmul(hidden_states, W_k)
value_states = np.matmul(hidden_states, W_v)

# Run the attention
context, weights = scaled_dot_product_attention(query_states, key_states, value_states)


print("Attention weights: " , weights)
print("Attention weights: " , weights.shape)

print("Context: ", context.shape )




## The projection Function


In [15]:
import numpy as np

def compute_qkv_states(hidden_states, weight_q, weight_k, weight_v, bias_q=None, bias_k=None, bias_v=None):
  """
    Projects input embeddings into Q, K, and V matrices.

    Args:
        hidden_states: Input embeddings of shape(batch_size, seq_len, model_dim)
        weight_q: Query Weight matrix of shape (model_dim, head_dim * num_heads)
        weight_k: key weight matrix of shape (model_dim, head_dim, * num_heads)
        weight_v: Value weight matrix of shape (mnodel_dim, head_dim * num_heads)
  """

  # Linear projection: Hidden states @ Weight matrix
  # Using np.dot or np.matmul (matrix multiplication)
  query_states= np.matmul(hidden_states, weight_q)
  key_states= np.matmul(hidden_states, weight_k)
  value_states = np.matmul(hidden_states, weight_v)

  # Add biases if they exist (common in BERT/GPT-2, optional in some newer models)
  if bias_q is not None:
    query_states += bias_q
  if bias_k is not None:
    key_states += bias_k
  if bias_v is not None:
    value_states += bias_v

  return query_states, key_states, value_states



## The Multi-Head Attention
To make this real we have to move beyond simple 3D tensors. In Multi-Head Attention (MHA), we take our projected Q, K, and V matrices and "chop" them into multiple heads so the model can look at the sequence from different "prespectives" simultaneously.




In [16]:
# The multi-head split
"""
The trick here is the reshape and transpose. We need to move "heads" dimension to the second
position so that the matrix multiplication happens correctly for each head independently
"""

import numpy as np

def split_heads(x, num_heads):
  """
  Splits the last dimension into (num_heads, head_dim).
  Input shape: [batch_size, seq_len, model_dim]
  Output shape: [batch_size, num_heads, seq_len, head_dim]
  """

  batch_size, seq_len, model_dim = x.shape
  head_dim = model_dim// num_heads

  # 1. Reshape to seperate  the heads
  x = x.reshape(batch_size, seq_len, num_heads, head_dim)

  # 2. transpose to [batch_size, num_heads, seq_len, head_dim]
  # This allows us to perform attention on each head in parallel
  return x.transpose(0, 2, 1, 3)


In [17]:
# The Concatenation (Merge)
"""
After running attention on each head, we get a result for each one. we need to glue them back
together and pass them through one final linear layer (Wo) to mix the information between heads.
"""

def merge_heads(x):
  """
  The inverse of split heads
  Input shape: [batch_size, num_heads, seq_len, head_dim]
  Output shape: [batch_size, seq_len, model_dim]
  """
  batch_size, num_heads, seq_len, head_dim = x.shape
  model_dim = num_heads * head_dim

  # 1. Transpose back to [batch_size, seq_len, num_heads, head_dim]
  x = x.transpose(0, 2, 1, 3)

  # 2. Reshape (Concatenate)
  return x.reshape(batch_size, seq_len, model_dim)

## Sinusoidal Positional Encoding ( The "Classic")

Introduced in the original Transformer paper, this uses fixed sine and cosine functions of different frequencies. It allows the model to extrapolated to sequence lengths longer than those seen during training.



In [19]:
import numpy as np

def get_sinusoidal_encoding(seq_len, d_model):
  pos = np.arrange(seq_len)[:, np.newaxis]
  i = np.arrange(d_model)[np.newaxis, :]

  # Compute the denominator (angle rates)
  angle_rates = 1/np.power(10000, (2, * (i //2 )/d_model))
  angle_rads = pos * angle_rates

  # Apply sin to even indices: cos to odd indices
  sines = np.sin(angle_rads[:, 0::2])
  cosines = np.cos(angle_rads[:, 1::2])

  pos_encoding = np.zeros((seq_len, d_model))
  pos_encoding[:, 0::2] = sines
  pos_encoding[:, 1::2] = cosines

  return pos_encoding


## Learned Positional Embeddings ( The "BERT/GPT-2 Stlye" )
instead of a fixed formulam, we treat positions as parameters to be learned, just like word embeddings

In [18]:
def get_learned(seq_len, d_model):
  # In a real training loop, this would be a parameter updated via backprop.
  # Here we initialise it randomly

  learned_weights = np.random.randn(seq_len, d_model) * 0.02
  return learned_weights


### Rotary Posotional Embeddings ( The "ROPE/Llama Style")
RoPE is the modern standard. Instead of adding a vector to the embedding, it rotates the Query and Key vectors in 2D planes. This captures relative distance between tokens very effectively

In [20]:
def apply_rope(x, seq_len, d_model):
  """
  Simplified RoPE implememntation for a single head.
  x shape: (batch, seq_len, d_model)
  """

  # 1. Create  rotation frequencies
  inv_freq = 1.0/(10000 ** (np.arrange(0, d_model, 2)/d_model))
  t = np.arrage(seq_len)
  freqs = np.outer(t, inv_freq) #(seq_len, d_model/2)

  # 2. Convert to polar coordinated (sin/cos)
  emb = np.concatenate((freqs, freqs), axis=-1)
  cos, sin = np.cos(emb), np.sin(emb)

  # 3. Apply rotation: x_rotated = (x * cos) + (rotate_half(x) * sin)
  def rotate_half(x):
    x1, x2 = x[..., :d_model//2], x[..., d_model//2:]
    return np.concatenate((-x2, x1), axis=-1)
  return (x* cos) + (rotate_half(x) *sin)


## Layer Normalisation
In Transformer architecture, Layer Normalisation(LayerNorm) is the preferred choice because it normalises across the features of a single sample, making it independent of other samples in a batch. This is crucial for sequence of varying lengths

In [21]:
import numpy as np
def layer_norm(x, gamma, beta, eps=1e-5):
  """
  Standard Layer Normalisation
  x shape: (batch_size, seq_len, model_dim)
  """

  # Calculate mean and variance across the last dimension (axis=-1)
  mean = np.mean(x, axis=-1, keepdims=True)
  var = np.var(x, axis=-1, keepdims=True)

  # Standardise
  x_norm = (x.mean) /np.sqrt(var + eps)

  # Scale and shift using learned parameters (gamma and beta)
  return gamma * x_norm + beta


## Batch Normalisation

In BatchNorm, we calculate the mean and variance across the batch and sequence dimensions for each feature. This means the normalisation of one sentence depends on the other sentences in the same batch, which can be unstable in NLP

In [23]:
def batch_norm(x, gamma, beta, eps=1e-5):
  """__
  Standard Batch Normalisation
  x shape: (batch_size, seq+_len, model_dim)
  """

  # Calculate mean and vairance across bacth and sequence (axis=(0, 1))
  # We normalise each feature (model_dim) independently
  mean = np.mean(x, axis=(0, 1), keepdims=True)
  var = np.var(x, axis=(0, 1), keepdims=True)

  # Standardise
  x_norm = (x-mean)/np.sqrt(var+eps)
  return gamma * x_norm + beta